# Why Conditioning Doesn't Steer QED — A Failure-Mode Investigation

**Question:** is it actually *impossible* to map QED to SELFIES/SMILES, or is steering just weak as
implemented (in both `d3pm_ultimate_conditional_diffusion` and `dit_conditional_generation`)?

This notebook answers that **entirely from the saved Stage-1 ablation package** (`results/stage1_ablation/`)
— no model, no GPU, no Drive, no sampling. It loads the per-molecule generations from the QED-only and
multi-property models and dissects *why* property conditioning barely moves QED.

**Short answer, proved below:** *not impossible.* The same model already emits QED ~0.89 under best-of-N —
drug-like molecules are in its output distribution. What is missing is *on-demand control*, for identifiable
and quantifiable reasons.

> Run top to bottom. Outputs (5 figures + `derived_stats.json` + `findings.md`) are written to
> `results/steering_investigation/`. Requires only pandas / numpy / matplotlib / scipy (+ RDKit for §4).


## 0. Setup & provenance

Load the saved per-molecule generations, best-of-N pools, QC, and configs. Crucially, print **how the data
under analysis was produced** — the operating point matters for interpretation: the sampler ran at
**`cfg=1.0`, i.e. with no classifier-free-guidance amplification**, so what we call "steering" here is the
*raw conditional* with the guidance knob turned off.


In [ ]:
%matplotlib inline
import os, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats as ss

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
                     "font.size": 11, "axes.grid": True, "grid.alpha": 0.30,
                     "axes.axisbelow": True})

def _find(rel_parts, extra=()):
    """Locate a path by trying repo-root and notebooks/ relative candidates."""
    rel = Path(*rel_parts)
    cands = [Path.cwd()/rel, Path.cwd()/"notebooks"/rel,
             *[p/rel for p in Path.cwd().parents],
             *[p/"notebooks"/rel for p in Path.cwd().parents], *extra]
    for c in cands:
        if c.exists():
            return c.resolve()
    raise FileNotFoundError(f"could not locate {rel} from {Path.cwd()}")

ABL = _find(("results", "stage1_ablation"))
OUT = ABL.parent/"steering_investigation"; OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT/"figures"; FIG.mkdir(exist_ok=True)
print("ablation in :", ABL)
print("writing to  :", OUT)

MODELS = ["QED-only", "Multi-prop"]
COLORS = {"QED-only": "#1f77b4", "Multi-prop": "#d62728"}

permol = {"QED-only":   pd.read_csv(ABL/"csv"/"native_steering_permol_qedonly.csv"),
          "Multi-prop": pd.read_csv(ABL/"csv"/"native_steering_permol_multiprop.csv")}
bestN  = {"QED-only":   pd.read_csv(ABL/"csv"/"bestofN_qedonly_qed0.9.csv"),
          "Multi-prop": pd.read_csv(ABL/"csv"/"bestofN_multiprop_qed0.9.csv")}
qc     = pd.read_csv(ABL/"csv"/"qc_summary.csv")
norm   = json.load(open(ABL/"config"/"norm_stats.json"))
runc   = json.load(open(ABL/"config"/"run_config.json"))

TARGETS = list(runc["targets"])
QPRIOR_MU, QPRIOR_SD = norm["qed"]["mean"], norm["qed"]["std"]
SAMP = runc["sampler"]
derived = {"provenance": {"seed": runc.get("seed"), "sampler": SAMP,
                          "targets": TARGETS, "qed_prior_mean": QPRIOR_MU,
                          "qed_prior_std": QPRIOR_SD, "n_per_target": runc.get("n_steer")}}

print("\n--- provenance (how the data under analysis was generated) ---")
print(f"seed={runc.get('seed')}  sampler: cfg={SAMP['cfg']} temp={SAMP['temperature']} "
      f"noise={SAMP['noise']} steps={SAMP['steps']}")
print(f"=> cfg={SAMP['cfg']} means NO classifier-free-guidance amplification (raw conditional).")
print(f"targets={TARGETS}  |  {runc.get('n_steer')} molecules per target per model")
print(f"data QED prior: mean={QPRIOR_MU:.3f} std={QPRIOR_SD:.3f}")
for m in MODELS:
    print(f"  loaded {m}: {len(permol[m])} per-mol rows, {len(bestN[m])} best-of-N rows")


## 1. The phenomenon — weak & asymmetric steering

If conditioning worked, achieved mean QED would track the requested target (slope 1, points on the dashed
line). Instead the response is **compressed** (slope far below 1) and **asymmetric**: low targets are roughly
hit, high targets are badly undershot, and the multi-property model barely responds at all.


In [ ]:
print("\n================ SECTION 1: weak & asymmetric steering ================")
agg = {}
for m in MODELS:
    d = permol[m]
    g = d.groupby("target")["qed"].agg(meanQED="mean", stdQED="std", medQED="median")
    g["meanHA"] = d.groupby("target")["ha"].mean()
    agg[m] = g.reindex(TARGETS)

sec1 = {}
for m in MODELS:
    mq = agg[m]["meanQED"].values
    lr = ss.linregress(TARGETS, mq)
    r_tq = ss.pearsonr(permol[m]["target"], permol[m]["qed"])[0]
    spread = float(np.nanmax(mq) - np.nanmin(mq))
    sec1[m] = {"spread_meanQED": spread, "slope_meanQED_vs_target": float(lr.slope),
               "corr_target_qed_permol": float(r_tq),
               "achieved_at_0.9": float(agg[m].loc[0.9, "meanQED"]),
               "achieved_at_0.3": float(agg[m].loc[0.3, "meanQED"])}
    print(f"{m:11s} spread={spread:.3f}  slope={lr.slope:.3f} (ideal 1.0)  "
          f"corr(target,QED)={r_tq:+.3f}  | @0.3={sec1[m]['achieved_at_0.3']:.3f}  @0.9={sec1[m]['achieved_at_0.9']:.3f}")
derived["sec1_phenomenon"] = sec1

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for m in MODELS:
    ax[0].errorbar(TARGETS, agg[m]["meanQED"], yerr=agg[m]["stdQED"], marker="o",
                   capsize=3, lw=2, color=COLORS[m], label=m)
ax[0].plot([0.25, 0.95], [0.25, 0.95], "k--", lw=1.2, label="ideal (achieved = target)")
ax[0].set(xlabel="target QED (requested)", ylabel="achieved mean QED",
          title="Steering response: achieved vs requested", xlim=(0.25, 0.95), ylim=(0.15, 0.95))
ax[0].legend(fontsize=9)
for m in MODELS:
    ax[1].plot(TARGETS, agg[m]["meanQED"].values - np.array(TARGETS), marker="s",
               lw=2, color=COLORS[m], label=m)
ax[1].axhline(0, color="k", lw=1)
ax[1].set(xlabel="target QED (requested)", ylabel="achieved - target (residual)",
          title="Systematic undershoot grows with target")
ax[1].legend(fontsize=9)
fig.tight_layout(); fig.savefig(FIG/"fig1_steering_response.png"); plt.show()
print("saved fig1_steering_response.png")


## 2. Two gaps, not one — compression + a generation-quality gap

The conditional distribution *does* move with the target (a non-trivial Cohen's *d* between the extreme
targets), so steering is not zero. But two things cap it: the shift is **compressed** (only a few hundredths
of QED per 0.1 requested), and the whole output distribution is **anchored below the training-data prior**
(the model generates lower-QED molecules than the data on average). Control gap *and* generation-quality gap.


In [ ]:
print("\n================ SECTION 2: regression to the prior ================")
# optional true-data QED prior
prior_qed = None
try:
    train_csv = _find(("data", "molecules_200k_selfies_clean.csv"))
    prior_qed = pd.read_csv(train_csv, usecols=["qed"])["qed"].dropna().values
    print(f"loaded true-data QED prior: n={len(prior_qed)} mean={prior_qed.mean():.3f}")
except Exception as e:
    print("true-data prior overlay skipped:", e)

def cohend(a, b):
    na, nb = len(a), len(b)
    sp = math.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / sp if sp > 0 else float("nan")

sec2 = {}
xs = np.linspace(0, 1, 240)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True, sharey=True)
cmap = plt.cm.viridis(np.linspace(0, 0.92, len(TARGETS)))
for ax, m in zip(axes, MODELS):
    if prior_qed is not None:
        kde0 = ss.gaussian_kde(prior_qed)(xs)
        ax.fill_between(xs, kde0, color="0.75", alpha=0.5, label="training-data QED", zorder=1)
    d = permol[m]
    for c, t in zip(cmap, TARGETS):
        v = d.loc[d["target"] == t, "qed"].values
        ax.plot(xs, ss.gaussian_kde(v)(xs), color=c, lw=1.8)
        ax.axvline(t, color=c, ls=":", lw=1.0, alpha=0.8)
    ax.axvline(QPRIOR_MU, color="k", lw=1.6, label=f"data prior mean ({QPRIOR_MU:.2f})")
    lo = d.loc[d["target"] == 0.3, "qed"]; hi = d.loc[d["target"] == 0.9, "qed"]
    sec2[m] = {"cohend_0.9_vs_0.3": float(cohend(hi, lo)),
               "gen_overall_mean_qed": float(d["qed"].mean()),
               "shift_per_0.1_target": float(ss.linregress(TARGETS, agg[m]["meanQED"]).slope*0.1)}
    ax.set(title=f"{m}: QED distribution per target\n(d[0.9 vs 0.3]={sec2[m]['cohend_0.9_vs_0.3']:.2f}, "
                 f"gen mean={sec2[m]['gen_overall_mean_qed']:.2f})",
           xlabel="achieved QED", xlim=(0, 1))
    ax.legend(fontsize=8, loc="upper right")
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(0.3, 0.9))
cb = fig.colorbar(sm, ax=axes, fraction=0.025, pad=0.01); cb.set_label("target QED")
axes[0].set_ylabel("density")
fig.savefig(FIG/"fig2_prior_regression.png"); plt.show()
derived["sec2_prior_regression"] = sec2
for m in MODELS:
    print(f"{m:11s} Cohen d(0.9 vs 0.3)={sec2[m]['cohend_0.9_vs_0.3']:.2f}  "
          f"gen mean QED={sec2[m]['gen_overall_mean_qed']:.3f} (vs data prior {QPRIOR_MU:.3f})")
print("saved fig2_prior_regression.png")


## 3. The mechanism — a molecular-size crutch

How does the model change QED at all? Partly by changing **molecule size**. We quantify it with a partial
correlation `corr(target, QED | heavy-atoms)` and a mediation estimate (how much of the target->QED effect
survives once size is controlled). For QED-only, size mediates a meaningful share but real structure-level
control remains; the multi-property model **over-drives** the size lever and collapses to fragments at high
targets — which is exactly why its net QED steering is ~0.


In [ ]:
print("\n================ SECTION 3: the size-proxy mechanism ================")
def partial_corr(x, y, z):
    rxy = np.corrcoef(x, y)[0, 1]; rxz = np.corrcoef(x, z)[0, 1]; ryz = np.corrcoef(y, z)[0, 1]
    return rxy, (rxy - rxz*ryz) / math.sqrt((1-rxz**2)*(1-ryz**2))

sec3 = {}
for m in MODELS:
    d = permol[m]; t, q, h = d["target"].values, d["qed"].values, d["ha"].values
    r_tq, pr = partial_corr(t, q, h)
    b1 = np.linalg.lstsq(np.c_[np.ones_like(t), t], q, rcond=None)[0][1]
    b2 = np.linalg.lstsq(np.c_[np.ones_like(t), t, h], q, rcond=None)[0][1]
    # the mediation ratio 1 - b2/b1 is only interpretable when a net effect (b1) actually exists;
    # for the Multi-prop model b1 ~ 0 (no net QED steering) so the ratio is undefined.
    meaningful = abs(b1) > 0.05
    mediated = float(1 - b2/b1) if meaningful else None
    sec3[m] = {"corr_target_qed": float(r_tq), "corr_ha_qed": float(np.corrcoef(h, q)[0, 1]),
               "corr_target_ha": float(np.corrcoef(t, h)[0, 1]), "partial_corr_target_qed_given_ha": float(pr),
               "beta_target_alone": float(b1), "beta_target_given_ha": float(b2),
               "frac_mediated_by_size": mediated, "net_effect_meaningful": bool(meaningful)}
    med = f"size-mediated {mediated*100:.0f}%" if meaningful else "mediation n/a: net QED effect ~0 (size over-driven -> suppression)"
    print(f"{m:11s} r(t,QED)={r_tq:+.3f}  r(HA,QED)={sec3[m]['corr_ha_qed']:+.3f}  "
          f"r(t,HA)={sec3[m]['corr_target_ha']:+.3f}  partial r(t,QED|HA)={pr:+.3f}  "
          f"| beta {b1:+.3f}->{b2:+.3f} ({med})")
derived["sec3_size_proxy"] = sec3

fig, ax = plt.subplots(2, 2, figsize=(12.5, 9.5))
for j, m in enumerate(MODELS):
    d = permol[m]
    sc = ax[0, j].scatter(d["ha"], d["qed"], c=d["target"], cmap="viridis", s=10, alpha=0.5)
    lr = ss.linregress(d["ha"], d["qed"])
    xx = np.linspace(d["ha"].min(), d["ha"].max(), 50)
    ax[0, j].plot(xx, lr.intercept + lr.slope*xx, "r-", lw=2)
    ax[0, j].set(title=f"{m}: QED vs size  (R^2={lr.rvalue**2:.2f})",
                 xlabel="heavy atoms", ylabel="achieved QED")
    plt.colorbar(sc, ax=ax[0, j], fraction=0.046, pad=0.04).set_label("target")
for m in MODELS:
    ax[1, 0].plot(TARGETS, agg[m]["meanHA"], marker="o", lw=2, color=COLORS[m], label=m)
ax[1, 0].set(title="The size lever: mean molecule size vs target",
             xlabel="target QED", ylabel="mean heavy atoms"); ax[1, 0].legend(fontsize=9)
parts = [permol["Multi-prop"].loc[permol["Multi-prop"]["target"] == t, "ha"].values for t in TARGETS]
ax[1, 1].violinplot(parts, positions=TARGETS, widths=0.07, showmedians=True)
ax[1, 1].set(title="Multi-prop size distribution collapses at high target",
             xlabel="target QED", ylabel="heavy atoms")
fig.tight_layout(); fig.savefig(FIG/"fig3_size_proxy.png"); plt.show()
print("saved fig3_size_proxy.png")


## 4. Which QED components actually move (RDKit)

QED is RDKit's aggregate of eight Bickerton desirability terms. Re-computing them for the *already-generated*
molecules (no model run — just parsing the saved SMILES) shows which levers the model pulls as the target
rises. The takeaway corrects the simple "it's only size" story: the model shifts a **broad** set of
components (notably HBA, PSA, structural alerts, ALOGP), i.e. it learned a genuine but **weak** multi-lever
QED response — the limit is *magnitude*, not absence.


In [ ]:
print("\n================ SECTION 4: which QED components move (RDKit) ================")
sec4 = {"available": False}
try:
    from rdkit import Chem
    from rdkit.Chem import QED
    from rdkit import RDLogger
    RDLogger.DisableLog("rdApp.*")
    COMPS = ["MW", "ALOGP", "HBA", "HBD", "PSA", "ROTB", "AROM", "ALERTS"]
    comp_rows = []; n_ok = n_bad = 0
    for m in MODELS:
        for t, smi in zip(permol[m]["target"].values, permol[m]["smiles"].values):
            mol = Chem.MolFromSmiles(str(smi)) if isinstance(smi, str) else None
            if mol is None:
                n_bad += 1; continue
            try:
                p = QED.properties(mol); n_ok += 1
                comp_rows.append({"model": m, "target": t, "MW": p.MW, "ALOGP": p.ALOGP,
                                  "HBA": p.HBA, "HBD": p.HBD, "PSA": p.PSA, "ROTB": p.ROTB,
                                  "AROM": p.AROM, "ALERTS": p.ALERTS})
            except Exception:
                n_bad += 1
    comp = pd.DataFrame(comp_rows)
    rate = n_ok/(n_ok+n_bad) if (n_ok+n_bad) else 0
    sec4["available"] = True; sec4["parse_rate"] = float(rate); sec4["n_parsed"] = int(n_ok)
    print(f"RDKit parsed {n_ok}/{n_ok+n_bad} generated SMILES ({rate*100:.1f}%)")
    fig, ax = plt.subplots(2, 4, figsize=(16, 7.5))
    corr_t = {}
    for k, cname in enumerate(COMPS):
        a = ax.flat[k]
        for m in MODELS:
            sub = comp[comp["model"] == m]
            mean_by_t = sub.groupby("target")[cname].mean().reindex(TARGETS)
            a.plot(TARGETS, mean_by_t.values, marker="o", lw=1.8, color=COLORS[m], label=m)
            corr_t.setdefault(m, {})[cname] = float(ss.pearsonr(sub["target"], sub[cname])[0])
        a.set_title(f"{cname}  (r_QEDonly={corr_t['QED-only'][cname]:+.2f})", fontsize=10)
        a.set_xlabel("target QED")
    ax.flat[0].legend(fontsize=8)
    fig.suptitle("QED desirability components vs target — which levers the model actually moves", y=1.02)
    fig.tight_layout(); fig.savefig(FIG/"fig4_qed_components.png"); plt.show()
    sec4["corr_component_vs_target"] = corr_t
    print("saved fig4_qed_components.png")
    print("  QED-only |r(target,component)|, sorted:")
    for cname, r in sorted(corr_t["QED-only"].items(), key=lambda kv: -abs(kv[1])):
        print(f"    {cname:7s} {r:+.3f}")
except Exception as e:
    print("RDKit section skipped:", e)
derived["sec4_components"] = sec4


## 5. Capacity exists; only control is missing

The decisive point for "is it possible?". Best-of-N reranking pulls QED ~0.89 (and a top-50 median well into
drug-like territory) out of the **same** model whose native mean at target 0.9 is ~0.46. The drug-like
molecules are already *in the distribution* — conditioning simply fails to summon them on demand. So the
mapping exists; the missing piece is a control mechanism.


In [ ]:
print("\n================ SECTION 5: capacity exists, control doesn't ================")
sec5 = {}
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.8))
for m in MODELS:
    pool = bestN[m]["qed"].values
    top50 = np.sort(pool)[::-1][:50]
    native09 = permol[m].loc[permol[m]["target"] == 0.9, "qed"].values
    sec5[m] = {"bestN_pool_top": float(pool.max()), "bestN_top50_median": float(np.median(top50)),
               "native_mean_at_0.9": float(native09.mean()), "native_max_at_0.9": float(native09.max()),
               "control_gap_top_minus_native": float(pool.max() - native09.mean())}
    ax[0].plot(xs, ss.gaussian_kde(pool)(xs), lw=2, color=COLORS[m], label=f"{m} best-of-N pool")
    ax[0].plot(xs, ss.gaussian_kde(native09)(xs), lw=2, ls="--", color=COLORS[m],
               label=f"{m} native @target0.9")
ax[0].axvline(QPRIOR_MU, color="k", lw=1.2)
ax[0].set(title="Best-of-N reaches high QED the native knob never does",
          xlabel="QED", ylabel="density", xlim=(0, 1)); ax[0].legend(fontsize=8)
labels = ["native\nmean@0.9", "best-of-N\ntop-50 median", "best-of-N\npool top"]
xpos = np.arange(3)
for i, m in enumerate(MODELS):
    vals = [sec5[m]["native_mean_at_0.9"], sec5[m]["bestN_top50_median"], sec5[m]["bestN_pool_top"]]
    ax[1].bar(xpos + i*0.38, vals, width=0.38, color=COLORS[m], label=m)
ax[1].axhline(0.7, color="green", ls=":", lw=1.5, label="drug-like (0.7)")
ax[1].set(title="Same model: capacity (right) vs on-demand control (left)",
          ylabel="QED", xticks=xpos + 0.19); ax[1].set_xticklabels(labels)
ax[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG/"fig5_capacity_vs_control.png"); plt.show()
derived["sec5_capacity"] = sec5
for m in MODELS:
    print(f"{m:11s} native mean@0.9={sec5[m]['native_mean_at_0.9']:.3f}  "
          f"best-of-N top50 median={sec5[m]['bestN_top50_median']:.3f}  pool top={sec5[m]['bestN_pool_top']:.3f}")
print("saved fig5_capacity_vs_control.png")


## 6. Conclusion & save

**Is it impossible to map QED to molecules? No.** Conditioning-via-denoising trains the property as a nearly
*redundant side-channel* of a reconstruction loss, so the model only ever develops a weak, partly
size-mediated knob — and here it was even measured with guidance off (`cfg=1.0`). The capacity is real and
present (best-of-N proves it); what is missing is any mechanism that supplies a **direct property gradient**:

- **RL fine-tuning** (Phase 3) — reward = achieved property; the real steering engine.
- **Classifier / predictor guidance** at sampling — an external property gradient (frozen predictor available).
- **Selection / best-of-N** — already works; steering by rejection.

This is the empirical motivation for the rest of the project. The cell below saves `derived_stats.json` and a
written `findings.md` summarising every number above.


In [ ]:
print("\n================ SAVE ================")
json.dump(derived, open(OUT/"derived_stats.json", "w"), indent=2)
print("wrote derived_stats.json")

s1, s2, s3, s4, s5 = (derived["sec1_phenomenon"], derived["sec2_prior_regression"],
                      derived["sec3_size_proxy"], derived["sec4_components"], derived["sec5_capacity"])

def comp_table(m):
    if not s4.get("available"):
        return "_(RDKit unavailable — section skipped)_"
    rows = sorted(s4["corr_component_vs_target"][m].items(), key=lambda kv: -abs(kv[1]))[:5]
    return ", ".join(f"{c} {r:+.2f}" for c, r in rows)

md = f"""# Why conditioning does not steer QED — failure-mode analysis

**One-line answer:** mapping QED to SELFIES/SMILES is **not impossible**. The same model already emits
QED {s5['QED-only']['bestN_pool_top']:.3f} under best-of-N, so drug-like molecules are in its output
distribution. What is weak is **on-demand control**, and this analysis quantifies why.

Every number below is derived (read-only) from the saved Stage-1 ablation package — **no model was run**.

**Provenance of the analysed data:** seed {derived['provenance']['seed']}, MaskGIT sampler
cfg={SAMP['cfg']} / temp={SAMP['temperature']} / noise={SAMP['noise']} / steps={SAMP['steps']},
{derived['provenance']['n_per_target']} molecules per target per model. Note **cfg={SAMP['cfg']} = no
classifier-free-guidance amplification** — steering was measured on the raw conditional.

## Headline numbers

| metric | QED-only | Multi-prop |
|---|---:|---:|
| steering spread (max-min mean QED) | {s1['QED-only']['spread_meanQED']:.3f} | {s1['Multi-prop']['spread_meanQED']:.3f} |
| slope: achieved mean QED vs target (ideal 1.0) | {s1['QED-only']['slope_meanQED_vs_target']:.3f} | {s1['Multi-prop']['slope_meanQED_vs_target']:.3f} |
| corr(target, QED) | {s3['QED-only']['corr_target_qed']:+.3f} | {s3['Multi-prop']['corr_target_qed']:+.3f} |
| partial corr(target, QED given size) | {s3['QED-only']['partial_corr_target_qed_given_ha']:+.3f} | {s3['Multi-prop']['partial_corr_target_qed_given_ha']:+.3f} |
| corr(target, size) — the size lever | {s3['QED-only']['corr_target_ha']:+.3f} | {s3['Multi-prop']['corr_target_ha']:+.3f} |
| generated mean QED (data prior = {QPRIOR_MU:.2f}) | {s2['QED-only']['gen_overall_mean_qed']:.3f} | {s2['Multi-prop']['gen_overall_mean_qed']:.3f} |
| best-of-N pool top QED | {s5['QED-only']['bestN_pool_top']:.3f} | {s5['Multi-prop']['bestN_pool_top']:.3f} |
| best-of-N top-50 median QED | {s5['QED-only']['bestN_top50_median']:.3f} | {s5['Multi-prop']['bestN_top50_median']:.3f} |

## Findings

**1. Steering is weak and asymmetric.** Achieved mean QED tracks the target with slope
{s1['QED-only']['slope_meanQED_vs_target']:.2f} (QED-only) — far below the ideal 1.0 — while the
Multi-prop model has essentially zero net response (slope {s1['Multi-prop']['slope_meanQED_vs_target']:.2f},
corr {s3['Multi-prop']['corr_target_qed']:+.2f}). The low end is roughly hit
(QED-only @0.3 -> {s1['QED-only']['achieved_at_0.3']:.2f}); the high end is badly missed
(@0.9 -> {s1['QED-only']['achieved_at_0.9']:.2f}).

**2. Two gaps, not one.** The conditional distribution *does* shift (Cohen d[0.9 vs 0.3] =
{s2['QED-only']['cohend_0.9_vs_0.3']:.2f} for QED-only), but it is (a) **compressed** — sliding only
~{abs(s2['QED-only']['shift_per_0.1_target']):.2f} QED per 0.1 of requested change — and (b) **anchored low**:
generated QED averages {s2['QED-only']['gen_overall_mean_qed']:.2f}, well below the training prior
{QPRIOR_MU:.2f}. So there is a *generation-quality* gap on top of the *control* gap.

**3. It leans on a molecular-size crutch.** For QED-only, ~{s3['QED-only']['frac_mediated_by_size']*100:.0f}%
of the (weak) target->QED effect is mediated by molecule size, though real structure-level control remains
(partial corr {s3['QED-only']['partial_corr_target_qed_given_ha']:+.2f}). The Multi-prop model drives size
even harder (corr(target,size) {s3['Multi-prop']['corr_target_ha']:+.2f}) and **over-shrinks into fragments
at high targets**, cancelling its QED gains — net effect ~0.

**4. It moves several QED components, not just size.** RDKit decomposition of the generated molecules
(parse rate {s4.get('parse_rate', 0)*100:.0f}%) shows the QED-only model shifts a *broad* set of Bickerton
components with target — strongest: {comp_table('QED-only')}. So the model learned a genuine but weak
multi-lever QED response; the limitation is **magnitude**, not absence.

**5. Capacity exists; only control is missing.** The same model reaches QED
{s5['QED-only']['bestN_pool_top']:.2f} (pool top) / {s5['QED-only']['bestN_top50_median']:.2f}
(top-50 median) under best-of-N, versus a native mean of {s5['QED-only']['native_mean_at_0.9']:.2f} at
target 0.9. The drug-like molecules are *in the distribution* — conditioning just does not summon them on demand.

## Conclusion

Conditioning-via-denoising trains the property as a near-redundant side-channel of a reconstruction loss,
so the model develops only a weak, partly size-mediated control knob — and it was measured with guidance
off (cfg=1.0). The mapping is real and present in the model; the fix is any mechanism that supplies a
**direct property gradient**: RL fine-tuning (Phase 3), classifier/predictor guidance at sampling, or
selection (best-of-N). This is the empirical motivation for making RL the steering engine.

## Figures
- `figures/fig1_steering_response.png` — achieved vs requested QED, plus the residual.
- `figures/fig2_prior_regression.png` — per-target QED distributions vs the data prior.
- `figures/fig3_size_proxy.png` — QED-vs-size, the size lever, and the Multi-prop fragment collapse.
- `figures/fig4_qed_components.png` — which QED desirability components respond to target.
- `figures/fig5_capacity_vs_control.png` — best-of-N capacity vs native on-demand control.
"""
open(OUT/"findings.md", "w", encoding="utf-8").write(md)
print("wrote findings.md")
print("DONE — figures in", FIG)
